In [ ]:
# === ARRANQUE EN COLAB: arbol de carpetas de la sesion =====================
# Este cuaderno se escribio para correr desde la carpeta `notebook/` de su
# sesion, con ../data, ../figuras y ../resultados al lado. Colab arranca en
# /content y sin ese arbol, asi que aqui se recrea y nos situamos dentro: con
# eso, todas las rutas relativas del cuaderno funcionan igual que en local.
import os, sys

if "google.colab" in sys.modules:
    _RAIZ = "/content/S13_supervivencia"
    for _sub in ("notebook", "data", "figuras", "resultados"):
        os.makedirs(os.path.join(_RAIZ, _sub), exist_ok=True)
    os.chdir(os.path.join(_RAIZ, "notebook"))
    print("Colab: carpeta de trabajo en", os.getcwd())


# Sesión 13 — Análisis de supervivencia — Kaplan-Meier, log-rank, Cox y CLV

**Curso:** Herramientas para la Ciencia de Datos, Facultad de Negocios, UPC
**Programa:** Administración y Ciencia de Datos para Negocios
**Técnica:** modelado de *tiempo-hasta-evento* con datos censurados (`lifelines`).

> **Cuaderno = laboratorio de replicación.** Se reproduce, paso a paso y con la librería `lifelines`, la **regresión de Cox (1972)** sobre el dataset canónico **Rossi** (reincidencia), la curva de **Kaplan-Meier (1958)** con la prueba **log-rank**, el diagnóstico de **hazards proporcionales (Schoenfeld)** y una segunda réplica sobre **Veterans**; luego se lleva la técnica al negocio (**churn** y **CLV** con Telco).

> **Cómo se abre este cuaderno.** El curso lo distribuye por **Google Drive**: en la
> carpeta compartida, clic derecho sobre el archivo → *Abrir con* → *Google
> Colaboratory*. Conviene empezar por **Archivo → Guardar una copia en Drive** para
> conservar el trabajo. No se requiere cuenta de GitHub ni instalar nada en el equipo:
> los datos de la sesión viajan dentro del propio cuaderno.
> **Carpeta del curso en Drive (Pregrado):** https://drive.google.com/drive/folders/1-YJxRt0n-UZwQCu03Lls2LGUYz6KMsl2


## 1. Objetivos de aprendizaje
- Reconocer **datos censurados** y por qué la regresión clásica no los maneja.
- Estimar y comparar curvas de supervivencia con **Kaplan-Meier** y la prueba **log-rank**.
- Ajustar e interpretar la **regresión de Cox** (*hazard ratio*) verificando el supuesto de **hazards proporcionales**.
- Calcular el **Customer Lifetime Value (CLV)** a partir de las curvas de supervivencia y aplicarlo a **churn**.

## 2. Mapa de la sesión: nueve capítulos en dos clases

La sesión ocupa **dos clases**. Cada capítulo lleva un código —13.1 a 13.9— que es **el mismo** en el sílabo, en la guía del docente, en la guía de laboratorio y en las diapositivas, de modo que se pueda pasar de un material a otro sin traducir numeraciones.

**JUEVES — 145 min de contenido** (bloque A1 de 75, receso de 15, bloque A2 de 70; antes, 20 min de control sobre la Sesión 12)

| Cód. | Pregunta que responde | Dónde vive en este cuaderno |
|---|---|---|
| **13.1** | ¿Por qué la regresión falla cuando el cliente aún no ha abandonado? | Sin celdas: se abre en clase con el cliente que aún no ha abandonado |
| **13.2** | ¿Qué son la censura, la supervivencia y el riesgo instantáneo? | «Teoría guiada — censura, `S(t)` y `h(t)`» |
| **13.3** | ¿Cómo se estima la curva sin suponer una forma funcional? | «Teoría guiada — censura, `S(t)` y `h(t)`» |
| **13.4** | ¿Cómo se mide el efecto de una variable sobre el tiempo? | «Teoría guiada — censura, `S(t)` y `h(t)`» |
| **13.5** | ¿Se sostiene con datos reales? La réplica de Cox (1972) | «Regresión de Cox (1972) sobre Rossi» — laboratorio, pasos 0 a 5 |

**VIERNES — 120 min corridos**

| Cód. | Pregunta que responde | Dónde vive en este cuaderno |
|---|---|---|
| **13.6** | ¿Cuándo se puede confiar en un modelo de Cox? | «Supuestos: identificación y condiciones de validez» |
| **13.7** | ¿Cómo se verifica que el resultado es real? | «Verificación desde la base» |
| **13.8** | ¿Qué decisión habilita? Tiempo hasta el abandono y valor de vida del cliente | «Del paper al negocio — CLV y churn con Telco» — laboratorio, paso 6 |
| **13.9** | ¿Qué no se puede afirmar, y qué sigue en S14? | «Cierre» |

> El **control** de esta sesión se resuelve en aula, en la franja de 20 minutos del jueves siguiente, y cubre **los nueve capítulos**, de los dos días.


## Cómo leer este cuaderno

Este cuaderno no solo **corre**: **explica y descompone** cada paso. Los marcadores guían la lectura:

- **❓ Qué se quiere averiguar** — abre cada resultado importante: la pregunta que ese resultado contesta, qué decisión depende de ella y **qué significaría cada resultado posible, dicho antes de ver el número**. Conviene detenerse ahí y contestar mentalmente antes de la ejecución: un dato solo informa a quien traía una pregunta.
- **🔎 Qué hace este código** — antes de cada celda: qué va a calcular y por qué.
- **📖 Cómo se lee esta salida** — después de una salida numérica clave: cómo interpretarla en negocio.
- **💡 Intuición** y **⚠️ Alerta / supuesto** — matices y errores a evitar (censura dependiente, `HR ≠ RR`, borrar la variable que viola PH).
- **🖐️ Cálculo manual** — se reconstruye la mecánica (el **estimador producto-límite de Kaplan-Meier** sobre una tabla de riesgo) y se verifica contra la librería con `assert`.
- **🧱 Construcción desde cero** — se reconstruye el **hazard ratio** `HR = exp(β)` desde el coeficiente ajustado y se reproduce el contrato (`assert`).
- **✅ Verificación desde la base** — se **recomputa** el resultado clave desde los datos (HR_fin 0,6843; S(52); Schoenfeld p_age 0,0007) y se cruza con el Excel (`assert`).
- **🧮 Matemática en el cuerpo** — la fórmula (`S(t)=P(T>t)`; `h(t)`; `S(t)=e^{−H(t)}`; producto-límite; `h(t|x)=h₀(t)·e^{x'β}`; `HR=e^{β}`; log-rank `χ²`) donde se aplica.
- **📄 En el paper** — procedencia exacta del dataset/resultado (autor, año, publicación).

**Convención Excel.** Los resultados y pruebas se vuelcan a `resultados/S13_resultados.xlsx` (hoja `survival`, contrato `B2:B9`) y las **figuras de resultados se generan LEYENDO ese Excel**. La **verificación desde la base** y los **diagnósticos de supuestos NO escriben en el Excel**.

**Regla de oro.** El valor **operativo** es el que produce el venv/Excel; toda cifra publicada de los papers va **ETIQUETADA como ancla**, nunca mezclada como propia.


## Preparación del entorno — Setup (Sección 0 del cuaderno)

Una sola celda de instalación para **Google Colab** (versiones fijadas). En ejecución local se salta automáticamente con la etiqueta `SKIP-LOCAL`.

In [ ]:
# SKIP-LOCAL: solo Colab.
# Colab ya trae el nucleo cientifico (numpy, pandas, scipy, matplotlib, seaborn,
# scikit-learn, statsmodels, openpyxl) COMPILADO ENTRE SI. Reinstalarlo con las
# versiones del venv del curso ROMPE el entorno: scipy y statsmodels dejan de
# importar con "cannot import name '_slice' from 'numpy._core.umath'". Por eso
# aqui solo se instala lo que Colab NO trae.
import sys

if "google.colab" in sys.modules:
    %pip install -q lifelines

# Trazabilidad (sin reinstalar): versiones en uso frente a la matriz
# certificada del curso en la matriz de versiones certificada del curso. Si alguna difiere, las cifras
# pueden variar en los ultimos decimales; el metodo y las conclusiones no.
import importlib.metadata as _md

_CERTIFICADAS = {
    "openpyxl": "3.1.5",
    "pandas": "2.3.3",
}

print(f"{'paquete':18}{'en uso':14}{'certificada':14}estado")
for _p, _cert in _CERTIFICADAS.items():
    try:
        _v = _md.version(_p)
    except Exception:
        _v = "ausente"
    _estado = "=" if _v == _cert else "distinta (se respeta la de Colab)"
    print(f"{_p:18}{_v:14}{_cert:14}{_estado}")


**🔎 Qué hace este código.** Importa `lifelines` (KM, Cox, Weibull, log-rank, Schoenfeld), fija rutas portables (funcionan vía `nbconvert` desde `notebook/` y en Colab) y define la paleta visual UPC.

In [ ]:
import warnings
warnings.simplefilter("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lifelines
from lifelines.datasets import load_rossi
from lifelines import CoxPHFitter, KaplanMeierFitter, WeibullFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test, proportional_hazard_test

print("lifelines", lifelines.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

# --- Rutas (funcionan vía nbconvert desde notebook/ y en Colab) ---
NB_DIR = Path.cwd()
S13 = NB_DIR.parent if NB_DIR.name == "notebook" else NB_DIR
DATA, RESULTS, FIGS = S13 / "data", S13 / "resultados", S13 / "figuras"
for d in (RESULTS, FIGS):
    d.mkdir(parents=True, exist_ok=True)

# --- Estilo visual UPC ---
UPC_RED, UPC_MAROON, INK, GRISM = "#E4002B", "#9B1B30", "#2D2D2D", "#6E6E6E"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "font.size": 11,
    "axes.edgecolor": GRISM, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": INK, "ytick.color": INK, "axes.grid": True,
    "grid.color": "#E6E6E6", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "figure.autolayout": True,
})

### 📄 En el paper — procedencia de los tres datasets

- **Rossi (reincidencia)** — datos experimentales de **Rossi, P. H., Berk, R. A. & Lenihan, K. J. (1980).** *Money, Work, and Crime.* Academic Press. Empaquetados **verbatim** en lifelines (`lifelines.datasets.load_rossi()`, **offline**); popularizados por Allison (1995) y Fox & Weisberg. 432 ex-reclusos, 52 semanas de seguimiento.
- **Veterans (cáncer de pulmón)** — **Kalbfleisch, J. D. & Prentice, R. L. (1980).** *The Statistical Analysis of Failure Time Data.* Wiley. Se carga vía **Rdatasets** (`survival::veteran`, mirror byte-idéntico del paquete R `survival`; 137 pacientes) — `scikit-survival` no instala en Python 3.13, así que se usa el CSV de Rdatasets.
- **Telco churn (negocio)** — **sustituto de negocio** reusado de S09: mirror abierto del CSV oficial de IBM (7 043 clientes, 21 columnas).

**🔎 Qué hace este código.** Define los cargadores: `cargar_rossi()` (offline), `cargar_veterans()` y `cargar_telco()` (usan `data/` local; si no existe, descargan del origen oficial).

In [ ]:
# --- Cargadores de datos (usan data/ local; si no existe, descargan del origen oficial) ---
URL_VET = ("https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/"
           "master/csv/survival/veteran.csv")
URL_TELCO = ("https://raw.githubusercontent.com/treselle-systems/customer_churn_analysis/"
             "master/WA_Fn-UseC_-Telco-Customer-Churn.csv")

def cargar_rossi():
    # Empaquetado en lifelines (offline): datos originales de Rossi, Berk & Lenihan (1980).
    return load_rossi()

def cargar_veterans():
    p = DATA / "veterans_lung_cancer.csv"
    return pd.read_csv(p) if p.exists() else pd.read_csv(URL_VET)

def cargar_telco():
    p = DATA / "telco_churn.csv"
    return pd.read_csv(p) if p.exists() else pd.read_csv(URL_TELCO)

print("Cargadores listos: cargar_rossi(), cargar_veterans(), cargar_telco().")

## 13.2 a 13.4 — ¿Qué son la censura, la supervivencia y el riesgo instantáneo, cómo se estima la curva y cómo se mide el efecto de una variable? Teoría guiada — censura, `S(t)` y `h(t)` (Sección 1 del cuaderno)

**Dato censurado (por la derecha):** al terminar el estudio el evento **aún no ocurrió**; solo se sabe que el tiempo real es **mayor** que el observado. En Rossi, un ex-recluso **no** re-arrestado en 52 semanas está *censurado* en 52. En churn, un cliente todavía activo al cortar los datos está censurado en su antigüedad actual.

**Por qué la regresión clásica no sirve:** (a) no puede representar «> 52 semanas» (descarta o distorsiona los censurados, que suelen ser la mayoría), (b) el tiempo es no negativo y asimétrico, y (c) ignora que el riesgo cambia con el tiempo. La supervivencia modela **tiempo + censura** de forma conjunta (ver la guía de supuestos de la sesión, Parte 1 y punto 4.1).

#### 🧮 Matemática en el cuerpo — supervivencia `S(t)`, riesgo `h(t)` y su relación

- **Función de supervivencia:** $S(t) = P(T > t)$ — proporción que sigue sin el evento más allá de $t$. Es la **curva de retención**; monótona decreciente con $S(0)=1$.
- **Función de riesgo (hazard):** la tasa **instantánea** del evento en $t$ dado que se sobrevivió hasta $t$,

$$h(t) = \lim_{\Delta t \to 0} \frac{P(t \le T < t+\Delta t \mid T \ge t)}{\Delta t}.$$

 No es una probabilidad (puede superar 1).
- **Riesgo acumulado** $H(t) = \int_0^t h(u)\,du$ y la **identidad puente** que conecta ambas funciones:

$$S(t) = \exp\!\big(-H(t)\big).$$

- **Estimador producto-límite de Kaplan-Meier** (no paramétrico), que se reconstruye de forma manual más abajo:

$$\hat S(t) = \prod_{t_i \le t} \left(1 - \frac{d_i}{n_i}\right),$$

 con $d_i$ = eventos y $n_i$ = sujetos en riesgo justo antes de $t_i$. Los **censurados reducen** $n_i$ sin bajar el escalón.

Fuente: el glosario de la sesión puntos 3-5; la guía de supuestos de la sesión, Parte 2.

**❓ Qué se quiere averiguar.** Al cerrar la observación, tres de los seis sujetos siguen activos: no se sabe cuándo —ni si— les llegará el evento. ¿Qué se hace con ellos: se descartan, se cuentan como si ya hubieran fallado, o existe una tercera vía?

- **Qué decide:** decide de qué muestra se habla. En una base de clientes, quienes siguen activos suelen ser la mayoría —y además los más recientes—, así que la elección no afecta a un caso aislado sino a la mayor parte de la cartera.
- **Antes de mirar el resultado:** si se descartan, solo quedan quienes ya se fueron y la conclusión queda predeterminada: la retención estimada cae de forma marcada hacia 0, porque en la muestra que sobrevive al filtro **todos terminaron por irse**. Si se les asigna el evento en la fecha de corte, se les atribuye una baja que nunca ocurrió. La tercera vía es la de esta sesión: quien sigue activo aporta información **parcial** —«su tiempo real es mayor que el observado»— y esa información entra en el cálculo sin inventar el desenlace.

**🔎 Qué hace este código.** Arma una **minidemostración** de 6 sujetos (algunos censurados) para ver el dato censurado antes de tocar Rossi: quién tuvo el evento y quién sigue activo («> tiempo»).

In [ ]:
# Mini-demo: 6 sujetos, algunos censurados (EDA directo, datos ilustrativos)
demo = pd.DataFrame({
    "sujeto": list("ABCDEF"),
    "tiempo": [6, 12, 12, 24, 24, 24],   # semanas observadas
    "evento": [1, 1, 0, 1, 0, 0],        # 1 = ocurrió; 0 = censurado (sigue activo)
})
demo["estado"] = np.where(demo["evento"] == 1, "evento", "censurado (> tiempo)")
print(demo.to_string(index=False))
print(f"\nSi se descartaran los {int((demo['evento']==0).sum())} censurados o se tratasen como 'evento', "
      "la retención quedaría sesgada. Kaplan-Meier usa su información parcial.")

**📖 Cómo se lee.** Tres sujetos (A, B, D) tuvieron el evento; tres (C, E, F) están **censurados**: solo se sabe que su tiempo real es **mayor** que el observado. Descartarlos sesga la retención hacia abajo; tratarlos como «evento» la sesga hacia arriba. La solución es usar su información **parcial**.

**🔎 Qué hace este código.** Dibuja la curva `S(t)` de la minidemostración con Kaplan-Meier: la escalera solo baja en los **eventos**; las **cruces** marcan censuras (no bajan el escalón).

In [ ]:
# S(t) de la mini-demo con Kaplan-Meier (curva escalonada; las cruces son censuras)
kmf_demo = KaplanMeierFitter().fit(demo["tiempo"], demo["evento"], label="S(t) demo")
ax = kmf_demo.plot_survival_function(color=UPC_RED, ci_show=False)
ax.scatter(demo.loc[demo.evento == 0, "tiempo"],
           [kmf_demo.survival_function_at_times(t).iloc[0] for t in demo.loc[demo.evento == 0, "tiempo"]],
           marker="+", s=140, color=INK, zorder=5, label="censura")
ax.set_xlabel("tiempo (semanas)"); ax.set_ylabel("S(t)"); ax.set_ylim(0, 1.02)
ax.set_title("Curva de supervivencia escalonada (los censurados no bajan el escalón)")
ax.legend(); plt.show()

**📖 Lectura de negocio.** `S(t)` es directamente la **curva de retención**: `S(12 meses)` = fracción de clientes aún activos al año. El hazard responde a *«¿en qué momento es más peligroso perder al cliente?»*; un hazard alto temprano marca la **ventana crítica** de retención.

**💡 Intuición.** El escalón cae **solo** cuando ocurre un evento y su tamaño depende de cuántos siguen en riesgo (`dᵢ/nᵢ`). Por eso al final de la curva —con pocos en riesgo— cada evento produce un salto grande y muy incierto: la **cola es frágil**, no señal.

## 13.5 — ¿Se sostiene con datos reales? Replicación — Regresión de Cox (1972) sobre Rossi (Sección 2 del cuaderno)

### Paso 0 — Contexto del método
- **Cox, D. R. (1972).** *Regression Models and Life-Tables.* JRSS-B 34(2):187-220. Introdujo `h(t|x)=h₀(t)·exp(x'β)`: el efecto de las covariables es **multiplicativo** sobre el hazard y el **hazard base `h₀(t)` queda sin especificar**, estimando `β` por **verosimilitud parcial**. Es el método más citado de la estadística aplicada del siglo XX.
- **Kaplan & Meier (1958).** *Nonparametric Estimation from Incomplete Observations.* JASA 53(282):457-481. El **estimador producto-límite** de `S(t)` con censura y su compañera, la prueba **log-rank**.

### 📄 En el paper — qué se replica y con qué honestidad histórica (subsección 2.0)

> **Transparencia histórica.** El **método** seminal es Cox (1972); **Rossi** es el **dataset canónico de enseñanza** del modelo de Cox (Allison 1995; Fox & Weisberg), cuya salida reproduce **verbatim** la propia documentación de `lifelines`. Se replica la **Cox estándar sobre Rossi**, no «la tabla numérica del paper de 1972».

**Lo que se reproduce (operativo = venv, ancla = lifelines docs):** `fin` **HR ≈ 0,684** (protector), `age` **HR ≈ 0,944** (protector), `prio` **HR ≈ 1,096** (riesgo); Schoenfeld **`age` viola PH** (p ≈ 0,001). Detalle en la ficha de la sesión de réplica del paper, tabla de targets (Sección 5).

### Qué preguntaban Kaplan-Meier y Cox, y por qué usaron lo que usaron — Sección 0 del paper (subsección 2.0)

**💡 Antes de tocar los datos.** Una réplica sin esta pregunta se vuelve mecánica: se ejecutan celdas y sale un número. Lo que sigue explica **qué buscaban los autores** y **por qué eligieron cada pieza de su método**, que es de donde proviene el criterio para elegir un método propio mañana. *(Desarrollo completo con las citas de los originales: la ficha de la sesión de réplica del paper, «Sección 0».)*

**Ninguno de los dos papers se propuso lo que se supone.** Kaplan y Meier no quisieron dibujar una curva escalonada, y Cox no quiso construir un modelo de churn: los dos atacan la misma anomalía del dato. Kaplan y Meier la enuncian en la primera línea de su resumen: «the observation of the time of occurrence of the event of interest (called a death) may be prevented for some of the items of the sample by the previous occurrence of some other event (called a loss)» (p. 457). Eso es **la censura**: del sujeto censurado no se sabe nada completo, pero tampoco nada vacío — se sabe algo estricto y verdadero, que **su tiempo hasta el evento es mayor que el observado**. Y la decisión que dependía de la respuesta era concreta: Kaplan medía la vida de los tubos de vacío de los repetidores de cables telefónicos hundidos en el océano; Meier venía de la duración del cáncer. El mismo obstáculo apareció, por separado, en ingeniería y en medicina.

**Por qué la censura rompe los dos atajos intuitivos.** Descartar a los censurados sesga la supervivencia **hacia abajo**, porque quien sale sin evento es justo quien más tarda. Tratar la observación truncada como si el evento hubiera ocurrido ahí —contar la antigüedad de un cliente todavía activo como si fuese su vida final— declara eventos que nadie observó e infla la mortalidad; y la variante opuesta, dar por hecho que el censurado nunca vivirá el evento, sobreestima la retención. Ninguna de las tres lecturas usa lo único que el dato censurado sí contiene: que el sujeto **estuvo en riesgo y no falló** durante todo el tramo observado. Es lo que se verificará en la celda siguiente, donde la mayoría de la muestra de Rossi resulta estar censurada.

**La pregunta de Cox, catorce años después.** Kaplan y Meier resolvieron la supervivencia de **un** grupo. Cox se planteó la siguiente: cómo medir el efecto de **varias variables explicativas** sobre el tiempo hasta el evento sin renunciar a los censurados. Su resumen lo dice: la función de riesgo se toma como función de las covariables y de coeficientes desconocidos, «multiplied by an arbitrary and unknown function of time», y de ahí se obtiene «a conditional likelihood».

**Las tres decisiones de método.** **(a) Un producto de probabilidades condicionales, no una proporción.** Con censura, el denominador cambia a cada instante; la única cantidad que **todo** sujeto puede alimentar es local, `1 − dᵢ/nᵢ` — la probabilidad de no fallar en `tᵢ` dado que se llegó vivo hasta ahí. El censurado achica el conjunto en riesgo y luego desaparece del denominador **sin contar como muerte**: de ahí `S(t) = ∏(1 − dᵢ/nᵢ)`, que se reconstruye de forma manual más abajo. La alternativa descartada, los intervalos fijos del método actuarial, obligaba a repartir las pérdidas dentro de cada intervalo por convención. **(b) El riesgo, y no el tiempo, como respuesta.** El hazard está definido **condicionado a seguir vivo**, que es justo la información que aporta un censurado, y una forma de producto deja que cada covariable entre como un factor que multiplica el riesgo. **(c) El riesgo base sin especificar.** En `h(t|x) = h₀(t) exp(x′β)`, `h₀(t)` queda libre: **se gana** no tener que decidir la forma de la curva base; **se paga** que el efecto de cada covariable ya no puede depender del tiempo — el supuesto de **riesgos proporcionales**. La **verosimilitud parcial** es lo que hace viable ese trato: al conservar solo los factores que comparan al sujeto que falla contra los que estaban en riesgo, `h₀(t)` se cancela y `β` se estima sin conocerlo.

**⚠️ La diferencia con el paper, dicha de frente.** Rossi **no** es la tabla numérica de Cox (1972) — él ilustró la teoría con ejemplos propios. Rossi es el dataset canónico de enseñanza del modelo, con la ayuda financiera **aleatorizada** y con la censura estructural y mayoritaria que reproduce el problema de 1958. Y hay una segunda advertencia que no debe suavizarse: el intervalo del HR de `fin` **roza 1**, de modo que la lectura honesta es «hay efecto y va en la dirección esperada, pero se financia con evaluación adicional», nunca «queda demostrado». El otro resultado obligatorio, la violación de riesgos proporcionales en `age`, tampoco es un accidente del dataset: es el **examen del precio** que se pagó en la decisión (c), y por eso el diagnóstico de Schoenfeld cierra la réplica.


### Paso 1 — Cargar Rossi y verificar la censura (A1-A2)

**🔎 Qué hace este código.** Carga Rossi (offline), cuenta **observaciones**, **eventos** (re-arresto) y **censurados**, y verifica la proporción censurada — el cortafuegos A1-A2 antes de modelar.

In [ ]:
rossi = cargar_rossi()
n_obs = rossi.shape[0]
n_ev = int(rossi["arrest"].sum())
n_cens = int((rossi["arrest"] == 0).sum())
print(f"Rossi: {n_obs} observaciones x {rossi.shape[1]} columnas")
print(f"Eventos (re-arresto, arrest==1): {n_ev}   |   Censurados: {n_cens}  "
      f"({n_cens/n_obs:.0%} de la base)")
print("\nColumnas: week (semana del arresto/censura), arrest (1=evento), "
      "fin (ayuda financiera), age, race, wexp, mar, paro, prio (condenas previas)")
rossi.head()

**📖 Cómo se lee (A1-A2).** El **74 %** de la muestra está **censurado** (no re-arrestado en 52 semanas): descartar esos casos o tratarlos como «nunca reincidirá» sesgaría todo.

**⚠️ Alerta.** Evento = `arrest`, duración = `week`. **Invertirlos rompería el análisis en silencio** (`REPLICACION_PAPER.md`, Sección 8). El supuesto raíz es la **censura no informativa**: la censura aquí es **administrativa** (corte a las 52 semanas), plausiblemente no informativa (`SUPUESTOS_S13.md`, punto 1.1).

### Paso 2 — Kaplan-Meier y log-rank por ayuda financiera (T5)

**❓ Qué se quiere averiguar.** ¿La ayuda financiera a la salida de prisión retrasa la reincidencia, o los dos grupos se comportan igual?

- **Qué decide:** `fin` es la única variable de la muestra sobre la que cabe actuar por política pública; el resto —edad, antecedentes— se observa, no se manipula. De esta comparación depende que el programa de ayuda tenga o no un argumento empírico.
- **Antes de mirar el resultado:** hay dos lecturas distintas y conviene no confundirlas. La **mediana**: si menos de la mitad reincide en 52 semanas, la curva nunca cruza 0,5 y la mediana sale **indefinida** —`inf` no es un fallo del código, es la respuesta honesta a una pregunta que los datos no alcanzan a contestar. Y la **separación**: si `S(52)` saliera casi idéntica en ambos grupos, la ayuda no cambiaría nada; si separa unos puntos de retención, queda por decidir si esa distancia es real (log-rank) y si compensa el costo del programa.

**🔎 Qué hace este código.** Estima la curva `S(t)` **global** (y su mediana) y luego **dos curvas** por ayuda financiera (`fin`); lee `S(52)` por grupo y las superpone.

In [ ]:
kmf = KaplanMeierFitter()

# Curva global
kmf.fit(rossi["week"], rossi["arrest"], label="global")
print("Mediana de supervivencia global:", kmf.median_survival_time_,
      "(inf = la curva no cruza 0,5 en el horizonte: <50 % reincide)")

# Dos grupos por ayuda financiera
g0 = rossi[rossi["fin"] == 0]   # sin ayuda
g1 = rossi[rossi["fin"] == 1]   # con ayuda
s0 = float(KaplanMeierFitter().fit(g0["week"], g0["arrest"]).survival_function_at_times(52).iloc[0])
s1 = float(KaplanMeierFitter().fit(g1["week"], g1["arrest"]).survival_function_at_times(52).iloc[0])
print(f"\nS(52 semanas) sin ayuda (fin=0): {s0:.4f}")
print(f"S(52 semanas) con ayuda (fin=1): {s1:.4f}")
print(f"Diferencia (con - sin): {s1 - s0:+.4f}  -> el grupo con ayuda sobrevive mejor")

# Plot inline por grupo
ax = KaplanMeierFitter().fit(g0["week"], g0["arrest"], label="sin ayuda (fin=0)").plot_survival_function(color=INK)
KaplanMeierFitter().fit(g1["week"], g1["arrest"], label="con ayuda (fin=1)").plot_survival_function(ax=ax, color=UPC_RED)
ax.set_xlabel("semanas"); ax.set_ylabel("S(t)"); ax.set_ylim(0.6, 1.01)
ax.set_title("Retención sin reincidencia por ayuda financiera (Rossi)"); plt.show()

**📖 Cómo se lee.** La **mediana global es `inf`**: la curva nunca cruza 0,5 en 52 semanas (menos de la mitad reincide), así que la mediana es **indefinida**, no un error. A 52 semanas, el grupo **con ayuda** retiene mejor (`S(52) ≈ 0,78` vs. `0,69` sin ayuda): +0,083 de retención.

**🔎 Qué hace este código.** Contrasta si las dos curvas difieren con la prueba **log-rank** (χ² y p-valor).

In [ ]:
lrt = logrank_test(g0["week"], g1["week"], g0["arrest"], g1["arrest"])
print(f"Log-rank por 'fin':  chi2 = {lrt.test_statistic:.3f}   p = {lrt.p_value:.4f}")

**📖 Lectura de negocio (T5).** El log-rank queda en el **límite de significancia** (`p ≈ 0,0501`): la diferencia es real pero moderada, coherente con el *hazard ratio* de `fin` < 1 que se obtiene enseguida. El log-rank dice **si** difieren, no **cuánto** — el «cuánto» lo da Cox.

**⚠️ Alerta.** El log-rank tiene **máxima potencia cuando los hazards son proporcionales** (las curvas no se cruzan). Si dos curvas **se cruzan**, las diferencias tempranas y tardías se cancelan y un p alto **no** significa «curvas iguales» (`SUPUESTOS_S13.md`, punto 2.2; drill 2).

#### 🧮 Matemática en el cuerpo — el producto-límite y el log-rank

**Kaplan-Meier** multiplica, en cada tiempo de evento $t_i$, la fracción que **sobrevive** ese instante:

$$\hat S(t) = \prod_{t_i \le t} \left(1 - \frac{d_i}{n_i}\right).$$

**Log-rank:** en cada $t_i$ compara los eventos **observados** $O_{ij}$ del grupo $j$ con los **esperados** $E_{ij}$ bajo $H_0$ (curvas iguales), y suma:

$$\chi^2 = \frac{\big(\sum_i (O_{i1}-E_{i1})\big)^2}{\sum_i V_i} \;\sim\; \chi^2_{g-1},$$

con $g$ grupos. Da un **p-valor, no un tamaño de efecto**. Fuente: el glosario de la sesión puntos 5-6.

#### 🖐️ Cálculo manual — el estimador producto-límite de Kaplan-Meier

**🔎 Qué hace este código.** Reconstruye **de forma manual** la tabla de riesgo de la minidemostración (para cada tiempo de evento: $n_i$ en riesgo y $d_i$ eventos), aplica $\hat S(t)=\prod(1-d_i/n_i)$ y **verifica con `assert`** que coincide (a tolerancia numérica) con `KaplanMeierFitter` de lifelines. Muestra además que un **censurado reduce** $n_i$ pero **no** baja el escalón.

In [ ]:
# 🖐️ HAZLO A MANO: producto-limite de Kaplan-Meier sobre una tabla de riesgo pequena (assert vs lifelines)
t = demo["tiempo"].values
e = demo["evento"].values
tiempos_evento = sorted(set(t[e == 1]))          # solo instantes con evento bajan el escalon

tabla, S = [], 1.0
for ti in tiempos_evento:
    n_i = int((t >= ti).sum())                   # en riesgo JUSTO antes de ti (censurados incluidos)
    d_i = int(((t == ti) & (e == 1)).sum())      # eventos en ti
    S *= (1 - d_i / n_i)                          # producto-limite
    tabla.append({"t": ti, "n_i (en riesgo)": n_i, "d_i (eventos)": d_i,
                  "factor (1-di/ni)": round(1 - d_i / n_i, 4), "S(t) a mano": round(S, 4)})
tabla = pd.DataFrame(tabla)
print(tabla.to_string(index=False))

# Cruce contra lifelines en cada tiempo de evento
kmf_check = KaplanMeierFitter().fit(demo["tiempo"], demo["evento"])
S_lib = {ti: float(kmf_check.survival_function_at_times(ti).iloc[0]) for ti in tiempos_evento}
print("\nS(t) lifelines:", {ti: round(v, 4) for ti, v in S_lib.items()})
for _, fila in tabla.iterrows():
    assert abs(fila["S(t) a mano"] - S_lib[fila["t"]]) < 1e-4, ("desajuste en t=", fila["t"])
print("\nassert OK: S(t)=prod(1-di/ni) a mano == KaplanMeierFitter (tolerancia 1e-4).")
print("Nota: el censurado en t=12 baja n_i de 5 a 3 para t=24, pero NO produce escalon.")

**📖 Cómo se lee.** El estimador **no es una caja negra**: es una simple multiplicación de fracciones de supervivientes. El **censurado en t=12** no baja la curva, pero **reduce** el conjunto en riesgo para el evento siguiente — esa es la mecánica exacta con la que la censura entra en el cálculo sin sesgar.

### Paso 3 — Regresión de Cox: hazard ratios (T1-T3, A3-A5)

**❓ Qué se quiere averiguar.** ¿Cuánto sube o baja cada característica el riesgo instantáneo de reincidir, con las demás constantes? Y, sobre todo, ¿qué pregunta **no** contesta esa cifra?

- **Qué decide:** el hazard ratio ordena las palancas. Con él se decide sobre qué variable interviene un programa, bajo un criterio triple —significativa, de magnitud apreciable y **accionable**—: `prio` puede ser la más significativa de todas y aun así no ser intervenible.
- **Antes de mirar el resultado:** un `HR < 1` marca factor protector, `HR > 1` factor de riesgo, y un IC 95 % que **cruza el 1** deja a la variable sin evidencia. Conviene fijar antes lo que el HR **no** dirá: no es una probabilidad, no es un riesgo acumulado y no responde «¿cuántas semanas más aguanta este sujeto?». Un `HR ≈ 0,68` significa que en cada instante ese grupo corre el 68 % del riesgo del otro, **no** que dure un 32 % más de tiempo. El tiempo que queda se lee en la curva `S(t)`, nunca en el HR.

**🔎 Qué hace este código.** Ajusta `CoxPHFitter` con las 7 covariables (sin transformar, sin penalización) y muestra la tabla de **hazard ratios** con IC 95 %, la **concordancia** (C-index), la log-verosimilitud parcial y el test de razón de verosimilitud.

In [ ]:
cph = CoxPHFitter().fit(rossi, duration_col="week", event_col="arrest")
tabla = cph.summary[["coef", "exp(coef)", "se(coef)",
                     "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]].round(4)
tabla.columns = ["coef(beta)", "HR", "se", "HR_low95", "HR_up95", "p"]
print(tabla.to_string())
print(f"\nn = {n_obs}  eventos = {n_ev}  censurados = {n_cens}")
print(f"Concordancia (C-index) = {cph.concordance_index_:.4f}")
print(f"Log-verosimilitud parcial = {cph.log_likelihood_:.2f}")
lr = cph.log_likelihood_ratio_test()
print(f"Test de razon de verosimilitud = {lr.test_statistic:.2f} (7 gl)  p = {lr.p_value:.2e}")

**📖 Lectura de negocio (T1-T3).**
- **`fin` HR ≈ 0,68** → recibir ayuda financiera **reduce** el hazard de reincidencia ~**32 %** (protector, accionable vía política de ayuda).
- **`age` HR ≈ 0,94** → cada año más de edad reduce el hazard ~**6 %**.
- **`prio` HR ≈ 1,10** → **cada condena previa multiplica** el hazard por 1,10 (**+10 % por antecedente**; efecto *multiplicativo*: 3 condenas → `1,10³ ≈ 1,33`). Es muy significativo pero **no accionable**.

El **C-index ≈ 0,64** indica discriminación **modesta** (fenómeno social difícil de predecir). Se prioriza por **significancia + magnitud del HR + accionabilidad**: `fin` es el candidato a intervención.

**⚠️ Alerta.** Comunicar `exp(coef)` (HR), **no** el coeficiente crudo `β`; el IC del HR que **cruza 1** = no significativo; el HR es **condicional** a las demás covariables y **presupone PH** (`SUPUESTOS_S13.md`, punto 3.4).

#### 🧮 Matemática en el cuerpo — el modelo de Cox y el hazard ratio

Cox modela el hazard como el producto de un **base sin especificar** y un factor exponencial de las covariables:

$$h(t \mid x) = h_0(t)\,\exp\!\big(x'\beta\big) = h_0(t)\,\exp\!\big(\beta_1 x_1 + \dots + \beta_p x_p\big).$$

Al subir una covariable **una unidad** (resto constante), el hazard se **multiplica** por el **hazard ratio**:

$$\text{HR}_j = \exp(\beta_j).$$

$\beta$ se estima **sin** modelar $h_0(t)$ maximizando la **verosimilitud parcial**: en cada tiempo de evento, quien lo sufre «compite» contra su conjunto en riesgo $R(t_i)$,

$$L(\beta) = \prod_{i:\, \text{evento}} \frac{\exp(x_i'\beta)}{\sum_{j \in R(t_i)} \exp(x_j'\beta)}.$$

Fuente: el glosario de la sesión puntos 7-8; `SUPUESTOS_S13.md`, Parte 3.

#### 🧱 Construcción desde cero — el hazard ratio `HR = exp(β)`

**🔎 Qué hace este código.** (1) Toma los coeficientes ajustados `cph.params_` y calcula `HR = exp(β)` **de forma manual** y verifica con `assert` que igualan la columna `exp(coef)` de lifelines. (2) Ilustra numéricamente la **verosimilitud parcial**: sobre un conjunto en riesgo pequeño, la contribución de quien tuvo el evento es `exp(xᵢ'β) / Σ exp(xⱼ'β)` — una probabilidad entre 0 y 1.

In [ ]:
# 🧱 CONSTRUYE DESDE CERO: HR = exp(beta) a mano (assert vs lifelines) + ilustracion de la verosimilitud parcial
beta = cph.params_                               # coeficientes ajustados (log-hazard ratios)
hr_mano = np.exp(beta)                           # HR = exp(beta)
hr_lib = cph.summary["exp(coef)"]
comp = pd.DataFrame({"beta": beta.round(4), "HR a mano = exp(beta)": hr_mano.round(4),
                     "HR lifelines": hr_lib.round(4)})
print(comp.to_string())
assert np.allclose(hr_mano.values, hr_lib.values, atol=1e-9), "HR=exp(beta) debe igualar exp(coef)"
assert abs(np.exp(beta["fin"]) - 0.6843) < 0.01, "HR_fin debe ser ~0,6843"
print("\nassert OK: exp(beta) a mano == exp(coef) de lifelines; HR_fin == 0,6843.")

# Ilustracion de la verosimilitud parcial sobre un conjunto en riesgo sintetico (una covariable, beta_prio)
b = float(beta["prio"])
riesgo = pd.DataFrame({"sujeto": list("PQRS"), "prio": [0, 1, 3, 2]})
riesgo["exp(x*beta)"] = np.exp(riesgo["prio"] * b)
contrib_Q = riesgo.loc[1, "exp(x*beta)"] / riesgo["exp(x*beta)"].sum()   # Q tiene el evento
print(f"\nVerosimilitud parcial: si Q (prio=1) sufre el evento en un riesgo de 4, su contribucion")
print(f"= exp(x_Q*beta) / sum exp(x_j*beta) = {contrib_Q:.4f}  (mas antecedentes -> mas 'culpa' del evento)")
assert 0 < contrib_Q < 1, "una contribucion de verosimilitud parcial es una probabilidad en (0,1)" 

**📖 Cómo se lee.** El *hazard ratio* **no es un artefacto de la librería**: es literalmente `exp` del coeficiente ajustado. Y la verosimilitud parcial es una **competencia** dentro del conjunto en riesgo: quien tiene covariables de mayor riesgo (más antecedentes) recibe mayor peso cuando ocurre un evento, lo que empuja su `β` hacia arriba — sin necesidad de modelar el hazard base `h₀(t)`.

### Paso 4 — Diagnóstico de hazards proporcionales: Schoenfeld (T4, drill 3)

El modelo de Cox **asume** que el HR de cada covariable es **constante en el tiempo**. El test de Schoenfeld lo verifica.

**⚠️ Lectura invertida (el error de lectura central de la sesión).** Aquí un **p BAJO = viola PH** (malo), lo **opuesto** a la tabla de Cox, donde un p bajo es **bueno** (variable significativa). El mismo `age` tiene p ≈ 0,009 en la tabla de Cox (significativa) y p ≈ 0,0007 en Schoenfeld (viola PH).

**❓ Qué se quiere averiguar.** Todos los hazard ratios recién leídos suponen que el efecto de cada variable es **el mismo en la semana 5 que en la semana 45**. ¿Se sostiene ese supuesto en estos datos?

- **Qué decide:** decide si la tabla anterior se puede comunicar tal cual. Si una covariable incumple, su HR único promedia dos épocas distintas, y ese promedio puede no describir bien ninguna de las dos.
- **Antes de mirar el resultado:** aquí el p-valor se lee **al revés** que en la tabla de Cox. Un `p` **alto** dice que no hay evidencia contra el supuesto: el HR es constante y legítimo. Un `p` **bajo** dice que la covariable **viola** PH y su efecto cambia con el tiempo. La misma `age` resulta significativa en Cox (p ≈ 0,009, buena noticia) y violadora de PH (p ≈ 0,0007, mala noticia): el signo de la noticia depende de qué tabla se mire, y esa es la confusión más común de la sesión.

**🔎 Qué hace este código.** Corre `proportional_hazard_test` con la transformación de tiempo a **rangos** (`time_transform` en modo `rank`) —que **transforma el tiempo a rangos antes de correlacionar los residuos de Schoenfeld con el tiempo**, y es lo que produce el `p_age ≈ 0,0007`— por covariable, e identifica cuál **viola** PH; además calcula los **residuales de Schoenfeld escalados** de `age` (insumo de la figura de resultados). Nota: `proportional_hazard_test` de lifelines y `cox.zph` de R difieren en la transformación por defecto del tiempo.

In [ ]:
zph = proportional_hazard_test(cph, rossi, time_transform="rank")
print(zph.summary[["test_statistic", "p"]].round(4).sort_values("p").to_string())
p_age = float(zph.summary.loc["age", "p"])
print(f"\nage: p = {p_age:.4f}  ->  VIOLA el supuesto de PH (p < 0,01)")
print("wexp queda marginal (p ~ 0,007); el resto (fin/prio/mar/paro/race) cumple (p > 0,10).")

# Residuales de Schoenfeld escalados de 'age' vs. tiempo (para la figura de resultados)
sch = cph.compute_residuals(rossi, kind="scaled_schoenfeld")
sch_age = pd.DataFrame({"week": rossi.loc[sch.index, "week"].values,
                        "resid_age": sch["age"].values}).sort_values("week")
print(f"\nResiduales de Schoenfeld de 'age' calculados: {len(sch_age)} puntos (uno por evento).")

**📖 Lectura (T4).** **`age` viola PH** (`p ≈ 0,0007`): su efecto **cambia con el tiempo** (la edad protege **más a largo plazo**). Un HR único para `age` es un **promedio engañoso**. **No se borra la variable** (perdería información y reintroduciría confusión); se **corrige** (se muestra en la sección de supuestos, punto 7.4). `wexp` **también viola PH de forma marginal** (p ≈ 0,0068 < 0,05, la misma violación que `age` aunque menos severa, no un simple «roce» del umbral); el resto cumple holgadamente. Fuente: `SUPUESTOS_S13.md`, punto 3.1; `plantillas/guia_schoenfeld.docx`.

### Paso 5 — Modelo paramétrico de Weibull (mención) y extrapolación

**🔎 Qué hace este código.** Ajusta `WeibullFitter` a la serie global y lee el **parámetro de forma `ρ`** (crece/decrece/constante el hazard).

In [ ]:
wf = WeibullFitter().fit(rossi["week"], rossi["arrest"])
print(f"Weibull:  rho (forma) = {wf.rho_:.4f}   lambda (escala) = {wf.lambda_:.2f}")
print(f"rho = {wf.rho_:.2f} > 1  ->  hazard CRECIENTE (desgaste): el riesgo aumenta con el tiempo.")
print("A diferencia de Kaplan-Meier, Weibull ESPECIFICA h0(t) y permite EXTRAPOLAR "
      "mas alla del horizonte observado (util para proyectar CLV a varios anios con poca historia).")

**📖 Lectura.** `ρ ≈ 1,37 > 1` → hazard **creciente**. Kaplan-Meier no extrapola (se queda plano al final); cuando el negocio necesita **proyectar** retención más allá de los datos (p. ej., CLV a 5 años con 1 de historia), un paramétrico como Weibull da una curva suave extrapolable (`SUPUESTOS_S13.md`, punto 4.2). El ajuste fino de familias paramétricas es **[Avanzado]**.

### Paso 6 — Segunda réplica: Cox sobre Veterans (T6)

**📄 En el paper.** Veterans' Administration Lung Cancer (**Kalbfleisch & Prentice 1980**), vía Rdatasets `survival::veteran` (137 pacientes). `trt` es 1=estándar / 2=test (se recodifica a 0/1); `celltype` es categórica (dummies, `drop_first`).

**🔎 Qué hace este código.** Ajusta Cox sobre Veterans y lee el HR de `karno` (estado funcional) y la significancia de `trt` (tratamiento).

In [ ]:
vet = cargar_veterans()
print(f"Veterans: {vet.shape[0]} pacientes x {vet.shape[1]} columnas")
vet["trt2"] = (vet["trt"] == 2).astype(int)          # 1=estandar, 2=test -> 0/1
dfv = vet[["time", "status", "trt2", "karno", "age", "diagtime", "prior", "celltype"]].copy()
dfv = pd.get_dummies(dfv, columns=["celltype"], drop_first=True).astype(float)
cv = CoxPHFitter().fit(dfv, duration_col="time", event_col="status")
tv = cv.summary[["exp(coef)", "p"]].round(4)
tv.columns = ["HR", "p"]
print(tv.to_string())
hr_karno = float(cv.summary.loc["karno", "exp(coef)"])
p_trt = float(cv.summary.loc["trt2", "p"])
print(f"\nkarno HR = {hr_karno:.4f} (<1, protector)   |   trt p = {p_trt:.4f} (>0,10, NO significativo)")
print(f"Concordancia = {cv.concordance_index_:.4f}")

**📖 Lectura de negocio/clínica (T6).** Cada punto del **índice de Karnofsky** (mejor estado funcional) **reduce** el hazard de muerte ~3 % (`karno` HR ≈ 0,97). El **tratamiento nuevo `trt` NO es significativo** (`p ≈ 0,16 > 0,10`): una vez ajustado por estado funcional y tipo de tumor, no mejora la supervivencia. `celltype` sí importa (squamous, HR ≈ 0,30, mejor pronóstico). Mensaje: **ajustar por el estado basal** cambia la conclusión sobre el tratamiento.

## Transversal — Exportación a Excel (convención del curso) (Sección 4 del cuaderno)

Todos los resultados y pruebas van a `resultados/S13_resultados.xlsx`. La hoja **`survival`** fija los valores operativos en `B2:B9` (contrato que lee el material de referencia de la sesión); las figuras de resultados se generan **leyendo este Excel**.

**🔎 Qué hace este código.** **ESCRIBE** el Excel de contrato: hoja `survival` (operativos `B2:B9` + bloque de apoyo) y las hojas de tablas para las figuras (`cox_rossi`, `schoenfeld`, `schoenfeld_age`, `km_rossi_fin`, `km_telco`, `clv_telco`, `veterans_cox`). Es la **única** celda que escribe el Excel.

In [ ]:
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows

hr = cph.summary["exp(coef)"]
wb = Workbook()

# --- Hoja 'survival': celdas operativas B2:B9 + bloque de apoyo ---
ws = wb.active; ws.title = "survival"
ws["A1"] = "metrica"; ws["B1"] = "valor_operativo"
operativos = [
    ("HR_fin",              float(hr["fin"])),      # B2  (T1, <1)
    ("HR_age",              float(hr["age"])),      # B3  (T2, <1)
    ("HR_prio",             float(hr["prio"])),     # B4  (T3, >1)
    ("Schoenfeld_p_age",    p_age),                 # B5  (T4, <0,01)
    ("dif_S52_fin1_menos_fin0", s1 - s0),           # B6  (T5, >0)
    ("logrank_p_fin",       float(lrt.p_value)),    # B7  (T5)
    ("HR_karno_veterans",   hr_karno),              # B8  (T6, <1)
    ("p_trt_veterans",      p_trt),                 # B9  (T6, >0,10)
]
for i, (k, v) in enumerate(operativos, start=2):
    ws[f"A{i}"] = k; ws[f"B{i}"] = v

apoyo = [
    ("n_obs", n_obs), ("n_eventos", n_ev), ("n_censurados", n_cens),
    ("concordancia_cox", float(cph.concordance_index_)),
    ("LR_test", float(lr.test_statistic)), ("loglik_parcial", float(cph.log_likelihood_)),
    ("S52_fin0", s0), ("S52_fin1", s1),
    ("weibull_rho", float(wf.rho_)), ("weibull_lambda", float(wf.lambda_)),
]
r = 12
ws[f"A{r-1}"] = "APOYO";
for k, v in apoyo:
    ws[f"A{r}"] = k; ws[f"B{r}"] = v; r += 1

# --- Hojas de tablas para las figuras ---
def add_df(name, df):
    w = wb.create_sheet(name)
    for row in dataframe_to_rows(df, index=False, header=True):
        w.append(row)

# Cox Rossi (forest de HR)
cox_forest = cph.summary.reset_index()[["covariate", "coef", "exp(coef)",
              "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
cox_forest.columns = ["covariate", "coef", "HR", "HR_low95", "HR_up95", "p"]
add_df("cox_rossi", cox_forest.round(6))

# Schoenfeld: test por variable + residuales de age (figura)
add_df("schoenfeld", zph.summary.reset_index()[["covariate", "test_statistic", "p"]].round(6)
       if "covariate" in zph.summary.reset_index().columns
       else zph.summary.reset_index().rename(columns={"index": "covariate"})[["covariate", "test_statistic", "p"]].round(6))
add_df("schoenfeld_age", sch_age.round(6))

# KM Rossi por fin (curvas 0..52)
wk = np.arange(0, 53)
km_rossi_fin = pd.DataFrame({
    "week": wk,
    "S_fin0": KaplanMeierFitter().fit(g0["week"], g0["arrest"]).survival_function_at_times(wk).values,
    "S_fin1": KaplanMeierFitter().fit(g1["week"], g1["arrest"]).survival_function_at_times(wk).values,
})
add_df("km_rossi_fin", km_rossi_fin.round(6))

# Veterans
add_df("veterans_cox", cv.summary.reset_index()[["covariate", "exp(coef)", "p"]].round(6))

XLSX = RESULTS / "S13_resultados.xlsx"
wb.save(XLSX)
print("Guardado:", XLSX)
print("Hojas:", wb.sheetnames)

## Transversal — Figuras de resultados (leyendo el Excel) (Sección 5 del cuaderno)

Las figuras de **resultados** (KM por segmento, forest de HR, Schoenfeld, CLV) se generan **leyendo `S13_resultados.xlsx`**, no de los objetos en memoria (convención del curso).

**🔎 Qué hace este código.** Abre el Excel, define el helper `leer(hoja)` y dibuja la **Figura 1**: Kaplan-Meier de Rossi por ayuda financiera (lee `km_rossi_fin`).

In [ ]:
from openpyxl import load_workbook
wbf = load_workbook(XLSX, data_only=True)

def leer(hoja):
    return pd.DataFrame(wbf[hoja].values).pipe(
        lambda d: d.rename(columns=d.iloc[0]).drop(0).reset_index(drop=True))

# 1) KM Rossi por ayuda financiera
kr = leer("km_rossi_fin").astype(float)
fig, ax = plt.subplots(figsize=(7, 4.3))
ax.step(kr["week"], kr["S_fin0"], where="post", color=INK, label="sin ayuda (fin=0)")
ax.step(kr["week"], kr["S_fin1"], where="post", color=UPC_RED, label="con ayuda (fin=1)", lw=2)
ax.set_xlabel("semanas"); ax.set_ylabel("S(t)"); ax.set_ylim(0.6, 1.005)
ax.set_title("Kaplan-Meier por ayuda financiera (Rossi)"); ax.legend()
fig.savefig(FIGS / "S13_km_rossi_fin.png", bbox_inches="tight"); plt.show()

**🔎 Qué hace este código.** **Figura 2**: forest de *hazard ratios* de la Cox sobre Rossi (lee `cox_rossi`); en rojo los significativos (`p<0,05`), escala log, línea en `HR=1`.

In [ ]:
# 2) Forest de hazard ratios (Cox Rossi)
cf = leer("cox_rossi")
for col in ["HR", "HR_low95", "HR_up95", "p"]:
    cf[col] = cf[col].astype(float)
cf = cf.sort_values("HR")
y = np.arange(len(cf))
fig, ax = plt.subplots(figsize=(7, 4.5))
cols = [UPC_RED if p < 0.05 else GRISM for p in cf["p"]]
ax.errorbar(cf["HR"], y, xerr=[cf["HR"] - cf["HR_low95"], cf["HR_up95"] - cf["HR"]],
            fmt="o", ecolor=GRISM, elinewidth=1.5, capsize=3, mfc="white", mec="none")
ax.scatter(cf["HR"], y, color=cols, zorder=5, s=55)
ax.axvline(1.0, color=INK, ls="--", lw=1)
ax.set_yticks(y); ax.set_yticklabels(cf["covariate"]); ax.set_xscale("log")
ax.set_xlabel("Hazard ratio (escala log; IC 95%)  —  <1 protector, >1 riesgo")
ax.set_title("Regresión de Cox sobre Rossi: hazard ratios")
fig.savefig(FIGS / "S13_forest_hr_cox.png", bbox_inches="tight"); plt.show()

**🔎 Qué hace este código.** **Figura 3**: residuales de Schoenfeld de `age` vs. tiempo (lee `schoenfeld_age`); la recta de tendencia con **pendiente ≠ 0** es la firma visual de la violación de PH.

In [ ]:
# 3) Residuales de Schoenfeld de 'age' vs. tiempo (viola PH)
sa = leer("schoenfeld_age").astype(float).sort_values("week")
fig, ax = plt.subplots(figsize=(7, 4.3))
ax.scatter(sa["week"], sa["resid_age"], color=GRISM, s=22, alpha=0.7)
z = np.polyfit(sa["week"], sa["resid_age"], 1)
xx = np.linspace(sa["week"].min(), sa["week"].max(), 50)
ax.plot(xx, np.polyval(z, xx), color=UPC_RED, lw=2.2, label="tendencia (pendiente ≠ 0 → viola PH)")
ax.axhline(0, color=INK, ls="--", lw=1)
ax.set_xlabel("semanas"); ax.set_ylabel("residual de Schoenfeld escalado (age)")
ax.set_title("Diagnóstico de Schoenfeld: el efecto de 'age' cambia con el tiempo")
ax.legend()
fig.savefig(FIGS / "S13_schoenfeld_age.png", bbox_inches="tight"); plt.show()

## 13.7 — ¿Cómo se verifica que el resultado es real? ✅ Verificación desde la base (Sección 6 del cuaderno)

El Excel de contrato ya está escrito. Ahora se comprueba que **es producto de ejecutar el código sobre los datos**, no un registro aislado: se **recomputan** las magnitudes clave —`HR_fin`, la diferencia `S(52)`, la `p` de Schoenfeld de `age` y el `HR_karno` de Veterans— con un ajuste **fresco** sobre las bases originales y se **cruzan con el Excel** mediante `assert`. Refleja lo que hace el material de referencia de la sesión (que recomputa desde la base). **No toca el Excel.**

**🔎 Qué hace este código.** Re-ajusta Cox/KM/Schoenfeld sobre Rossi y Cox sobre Veterans desde cero, y cruza los cuatro con las celdas `B2`, `B6`, `B5` y `B8` de la hoja `survival` con `assert`.

In [ ]:
# ✅ VERIFICACION DESDE LA BASE: recomputa HR_fin, dif S(52), Schoenfeld p_age y HR_karno, y los cruza con el Excel (assert). NO toca el Excel.
wb_chk = load_workbook(XLSX, data_only=True)["survival"]
xl_hr_fin  = float(wb_chk["B2"].value)   # HR_fin
xl_p_age   = float(wb_chk["B5"].value)   # Schoenfeld p(age)
xl_difS52  = float(wb_chk["B6"].value)   # S(52) fin1 - fin0
xl_hr_karno= float(wb_chk["B8"].value)   # HR karno Veterans

# (1) HR_fin recomputado (Cox fresco sobre Rossi)
rossi_v = cargar_rossi()
cph_v = CoxPHFitter().fit(rossi_v, duration_col="week", event_col="arrest")
re_hr_fin = float(cph_v.summary.loc["fin", "exp(coef)"])

# (2) dif S(52) recomputada (KM fresco por fin)
a0, a1 = rossi_v[rossi_v["fin"] == 0], rossi_v[rossi_v["fin"] == 1]
re_s0 = float(KaplanMeierFitter().fit(a0["week"], a0["arrest"]).survival_function_at_times(52).iloc[0])
re_s1 = float(KaplanMeierFitter().fit(a1["week"], a1["arrest"]).survival_function_at_times(52).iloc[0])
re_difS52 = re_s1 - re_s0

# (3) Schoenfeld p(age) recomputado
re_p_age = float(proportional_hazard_test(cph_v, rossi_v, time_transform="rank").summary.loc["age", "p"])

# (4) HR_karno recomputado (Cox fresco sobre Veterans)
vet_v = cargar_veterans(); vet_v["trt2"] = (vet_v["trt"] == 2).astype(int)
dfv_v = pd.get_dummies(vet_v[["time","status","trt2","karno","age","diagtime","prior","celltype"]],
                       columns=["celltype"], drop_first=True).astype(float)
re_hr_karno = float(CoxPHFitter().fit(dfv_v, duration_col="time", event_col="status").summary.loc["karno", "exp(coef)"])

tabla_v = pd.DataFrame([
    ["HR_fin (Cox Rossi)",        re_hr_fin,   xl_hr_fin,   0.6843],
    ["dif S(52) fin1-fin0",       re_difS52,   xl_difS52,   0.0833],
    ["Schoenfeld p(age)",         re_p_age,    xl_p_age,    0.0007],
    ["HR_karno (Cox Veterans)",   re_hr_karno, xl_hr_karno, 0.9677],
], columns=["magnitud", "recomputado", "Excel", "ancla"])
print(tabla_v.to_string(index=False))

assert abs(re_hr_fin    - xl_hr_fin)   < 1e-6, "HR_fin recomputado != Excel B2"
assert abs(re_difS52    - xl_difS52)   < 1e-6, "dif S(52) recomputada != Excel B6"
assert abs(re_p_age     - xl_p_age)    < 1e-6, "Schoenfeld p(age) recomputado != Excel B5"
assert abs(re_hr_karno  - xl_hr_karno) < 1e-6, "HR_karno recomputado != Excel B8"
print("\nassert OK: recomputado desde la base == Excel (B2,B5,B6,B8) y consistente con las anclas.")

## 13.6 — ¿Cuándo se puede confiar en un modelo de Cox? Supuestos: identificación y condiciones de validez (Sección 7 del cuaderno)

> **Fuente única:** la guía de supuestos de la sesión. Esta sección **ejecuta diagnósticos** que **NO escriben en el Excel**: hacen plausible o contrastan cada supuesto y **nombran la corrección**. El caso central es la **violación de PH de `age`** (Schoenfeld p 0,0007): se **corrige**, no se borra la variable.

| Diagnóstico | Supuesto | Cómo se identifica | Cómo se decide / corrige | Fuente |
|---|---|---|---|---|
| 7.1 Censura no informativa | raíz (KM/log-rank/Cox) | mecanismo de censura; descartar vs. tratar-como-evento sesga | declarar el mecanismo; codificar bien evento/tiempo | Parte 1.1 |
| 7.2 KM + log-rank | potencia bajo PH | ¿las curvas se cruzan?; p en el límite (0,0501) | cribado + Cox; ponderadas/RMST si se cruzan | Parte 2.1-2.2 |
| 7.3 Hazards proporcionales | el supuesto central | Schoenfeld p<0,05 (`age` 0,0007); residuales con pendiente | estratificar / interacción con el tiempo / splines | Parte 3.1 |
| 7.4 Corrección de la violación | PH de `age` | HR único = promedio engañoso | estratificar (no borrar) / interactuar con `t` | Parte 3.1 |


### — Censura no informativa (el supuesto raíz) — capítulo 13.6 (subsección 7.1)

**🔎 Qué hace este código.** Ilustra por qué el supuesto raíz importa: sobre Rossi compara tres lecturas de `S(52)` — **(a)** descartar censurados, **(b)** tratar los censurados como si hubieran tenido el evento, **(c)** Kaplan-Meier (que usa la información parcial) — y verifica con `assert` que KM queda **entre** los dos sesgos. **No escribe el Excel.**

In [ ]:
# 7.1 Censura no informativa: ignorar la censura da una retencion absurda; KM la corrige (assert). NO toca el Excel.
horizon = 52
# En Rossi la censura es ADMINISTRATIVA: los 318 no re-arrestados quedan censurados en la semana 52.
# (a) Ignorar la censura = tomar el tiempo observado como 'vida completa': S(52) = fraccion con week>52.
s_ignora_censura = float((rossi["week"] > horizon).mean())          # -> 0 %: 'nadie sobrevive' al corte (absurdo)
# (b) Kaplan-Meier: usa la censura como informacion parcial (los censurados no bajan el escalon).
s_km = float(KaplanMeierFitter().fit(rossi["week"], rossi["arrest"]).survival_function_at_times(horizon).iloc[0])
print(f"S(52) ignorando la censura (tiempo = vida final) = {s_ignora_censura:.4f}   -> retencion absurda (0 %)")
print(f"S(52) Kaplan-Meier (censura no informativa)      = {s_km:.4f}   -> correcto ({s_km:.0%} sigue sin reincidir)")
print(f"(coincide con la fraccion censurada {int((rossi['arrest']==0).sum())}/{len(rossi)} = {(rossi['arrest']==0).mean():.4f})")
assert s_ignora_censura < s_km < 1.0, "KM debe corregir el sesgo de ignorar la censura"
print("\nassert OK: ignorar la censura da 0 % de retencion (absurdo); KM la fija en ~74 %. NO hay test de")
print("no-informatividad: se argumenta por el MECANISMO (censura administrativa a la semana 52, plausible).")

**📖 Cómo se lee.** Descartar a los censurados (mirar solo a quienes reincidieron) lleva la retención estimada hacia 0; Kaplan-Meier la sitúa en su valor correcto (~0,80) usando la información parcial. **No hay test** que confirme la no-informatividad: se argumenta por el **mecanismo** (aquí, censura **administrativa** a las 52 semanas, plausiblemente no informativa). El caso peligroso sería censura **dependiente** del pronóstico (`SUPUESTOS_S13.md`, punto 1.1).

### — Kaplan-Meier + log-rank (potencia bajo hazards proporcionales) — capítulo 13.3 (subsección 7.2)

**🔎 Qué hace este código.** Recalcula el log-rank por `fin` y verifica que las curvas **no se cruzan** en el horizonte (`S(t|fin=1) ≥ S(t|fin=0)` en todo `t`), condición bajo la que el log-rank conserva potencia. **No escribe el Excel.**

In [ ]:
# 7.2 KM + log-rank: las curvas no se cruzan -> el log-rank conserva potencia (assert). NO toca el Excel.
lrt_v = logrank_test(g0["week"], g1["week"], g0["arrest"], g1["arrest"])
wk = np.arange(1, 53)
S_f0 = KaplanMeierFitter().fit(g0["week"], g0["arrest"]).survival_function_at_times(wk).values
S_f1 = KaplanMeierFitter().fit(g1["week"], g1["arrest"]).survival_function_at_times(wk).values
n_cruces = int((np.sign(S_f1 - S_f0) < 0).sum())     # semanas donde con-ayuda queda por debajo
print(f"log-rank por 'fin': chi2 = {lrt_v.test_statistic:.3f}  p = {lrt_v.p_value:.4f}  (limite ~0,05)")
print(f"semanas (1..52) donde S(fin=1) < S(fin=0): {n_cruces}  -> las curvas NO se cruzan")
assert lrt_v.p_value < 0.06, "log-rank de fin queda cerca del limite 0,05"
assert n_cruces == 0, "si las curvas no se cruzan, el log-rank conserva potencia"
print("\nassert OK: curvas separadas sin cruce -> el p 0,0501 es interpretable como diferencia real moderada.")

**📖 Cómo se lee.** El log-rank roza el 5 % (`p ≈ 0,0501`): ni «claramente distintas» ni «iguales». Como las curvas **no se cruzan**, la prueba conserva potencia y el p es interpretable; **si se cruzaran**, se preferirían pruebas ponderadas (Gehan/Fleming-Harrington) o RMST (`SUPUESTOS_S13.md`, punto 2.2; drill 2).

### — Hazards proporcionales: `age` viola PH (el supuesto central) — capítulo 13.6 (subsección 7.3)

**🔎 Qué hace este código.** Recorre la salida de Schoenfeld: verifica con `assert` que **`age` viola** PH (`p < 0,01`) y que las covariables accionables (`fin`, `prio`) **cumplen** (`p > 0,10`), y mide la **pendiente** de los residuales de `age` vs. tiempo (la firma de la violación). **No escribe el Excel.**

In [ ]:
# 7.3 Schoenfeld: age viola PH; fin y prio cumplen; residuales de age con pendiente (assert). NO toca el Excel.
zph_v = proportional_hazard_test(cph, rossi, time_transform="rank")
p_por_var = zph_v.summary["p"].round(4)
print(p_por_var.sort_values().to_string())
p_age_v  = float(p_por_var.loc["age"])
p_fin_v  = float(p_por_var.loc["fin"])
p_prio_v = float(p_por_var.loc["prio"])

sa_v = cph.compute_residuals(rossi, kind="scaled_schoenfeld")
w_ev = rossi.loc[sa_v.index, "week"].values
pend = float(np.polyfit(w_ev, sa_v["age"].values, 1)[0])   # pendiente residual age vs tiempo
print(f"\np(age)={p_age_v:.4f} (<0,01 VIOLA)  p(fin)={p_fin_v:.4f}  p(prio)={p_prio_v:.4f}  (>0,10 cumplen)")
print(f"pendiente de los residuales de 'age' vs. tiempo = {pend:+.5f}  (!= 0 -> efecto cambia con el tiempo)")
assert p_age_v < 0.01, "age debe violar PH"
assert p_fin_v > 0.10 and p_prio_v > 0.10, "fin y prio deben cumplir PH"
assert abs(pend) > 0, "los residuales de age deben tener pendiente (efecto dependiente del tiempo)"
print("\nassert OK: age viola PH (p 0,0007); fin/prio cumplen. p BAJO en Schoenfeld = MALO (lectura invertida).")

**📖 Cómo se lee.** `age` viola PH: su efecto **cambia con el tiempo** (protege más a largo plazo), por eso los residuales tienen **pendiente**. Las variables **accionables** (`fin`, `prio`) cumplen, así que sus HR únicos son legítimos. **Recordatorio del error de lectura:** un p **bajo** en Schoenfeld es **malo** (viola), al revés que en la tabla de Cox.

### — La corrección: estratificar o interactuar con el tiempo (NO borrar la variable) — capítulo 13.6 (subsección 7.4)

**❓ Qué se quiere averiguar.** Una covariable incumple hazards proporcionales. ¿Qué se hace con ella?

- **Qué decide:** decide el modelo que se entrega. La salida más tentadora —borrar la variable que molesta— es también la peor: elimina información y puede reintroducir la confusión que esa variable controlaba, de modo que arregla el diagnóstico y estropea la estimación.
- **Antes de mirar el resultado:** la elección entre las dos rutas depende de para qué hace falta la variable. Si su HR **no** interesa y solo estorba, **estratificar** la absorbe en un hazard base por estrato: `age` deja de tener HR propio y las demás covariables deberían volver a cumplir PH. Si su efecto **sí** interesa, el término `age × t` **describe** cómo cambia ese efecto con el tiempo, en lugar de promediarlo en una cifra única. La comprobación de que cualquiera de las dos funcionó es la misma: repetir Schoenfeld y ver si la violación desapareció.

**🔎 Qué hace este código.** Aplica **dos correcciones** a la violación de `age` y verifica cada una: **(a) estratificar** por `age` binado (`CoxPHFitter(strata=...)` → un hazard base por estrato; `age` deja de tener HR y las restantes vuelven a cumplir PH) y **(b) interacción con el tiempo** (término `age × t` que **modela** cómo cambia el efecto). Ninguna **borra** la variable. **No escribe el Excel.**

In [ ]:
# 7.4 Correccion de la violacion de PH de 'age': (a) estratificar  (b) interaccion con el tiempo. NO toca el Excel.
# (a) ESTRATIFICAR por age binado -> un h0(t) por estrato; age se controla, no se estima su HR
rossi_str = rossi.copy()
rossi_str["age_bin"] = pd.qcut(rossi_str["age"], 3, labels=["joven", "medio", "mayor"])
cph_str = CoxPHFitter().fit(rossi_str.drop(columns=["age"]),
                            duration_col="week", event_col="arrest", strata=["age_bin"])
zph_str = proportional_hazard_test(cph_str, rossi_str.drop(columns=["age"]), time_transform="rank")
p_min_tras = float(zph_str.summary["p"].min())
print("(a) Estratificando por age (binado) — PH de las covariables restantes:")
print(zph_str.summary[["test_statistic", "p"]].round(4).sort_values("p").to_string())
print(f"    'age' ya no entra como covariable proporcional; min p restante = {p_min_tras:.4f} (>0,05 cumplen)")
assert "age" not in cph_str.summary.index, "al estratificar, age deja de tener HR"
assert p_min_tras > 0.05, "tras estratificar, las covariables restantes deben cumplir PH"

# (b) INTERACCION CON EL TIEMPO: age x t modela como cambia el efecto (CoxTimeVaryingFitter via formato long)
from lifelines import CoxTimeVaryingFitter
from lifelines.utils import to_episodic_format
rl = rossi.reset_index().rename(columns={"index": "id"})
long = to_episodic_format(rl, duration_col="week", event_col="arrest", id_col="id", time_gaps=13.0)
long["age_x_t"] = long["age"] * long["stop"]      # termino de interaccion age x tiempo
ctv = CoxTimeVaryingFitter().fit(long[["id","start","stop","arrest","fin","age","age_x_t","prio"]],
                                 id_col="id", event_col="arrest", start_col="start", stop_col="stop")
p_int = float(ctv.summary.loc["age_x_t", "p"])
print(f"\n(b) Interaccion age x t: coef = {ctv.summary.loc['age_x_t','coef']:+.5f}  p = {p_int:.4f}")
print("    Un termino age x t significativo confirma que el efecto de la edad CAMBIA con el tiempo:")
print("    se MODELA la trayectoria en vez de resumirla en un HR unico (o se estratifica).")
assert "age_x_t" in ctv.summary.index, "la interaccion con el tiempo modela el efecto dependiente del tiempo"
print("\nassert OK: la violacion de PH se CORRIGE (estratificar / interactuar con t), NO se borra la variable.")

**📖 Cómo se lee.** Las dos rutas resuelven la violación **sin descartar** `age`: **(a)** estratificar la absorbe en un hazard base por estrato (útil si su HR no interesa) y devuelve a las demás covariables a cumplir PH; **(b)** el término `age × t` **describe** la trayectoria del efecto (útil si sí interesa: «la edad protege más a largo plazo»). Borrar la variable perdería información y podría reintroducir confusión (`SUPUESTOS_S13.md`, punto 3.1; `plantillas/guia_schoenfeld.docx`). El coeficiente dependiente del tiempo fino y los splines son **[Avanzado]**.

## Práctica — Drills (para resolver) (Sección 8 del cuaderno)

Enunciados completos en `evaluacion/drills.docx` (se resuelven en parejas; entrega individual). Cada drill cubre un punto de la guía de supuestos de la sesión:

1. **Interpretar un *hazard ratio* de 1,8** (punto 3.4) en términos de negocio: ¿qué significa exactamente y qué **no** significa? Traducirlo a una acción de retención (comunicar `exp(coef)` + IC; multiplicatividad; HR ≠ probabilidad ≠ riesgo relativo acumulado).
2. **Comparar dos curvas de Kaplan-Meier con la prueba log-rank** (puntos 2.1-2.2): sobre Rossi por `fin` (`S(52)` 0,69 vs 0,78; χ² ≈ 3,84; p ≈ 0,05), decidir si difieren, interpretar la **dirección** y nombrar dos errores de lectura (curvas que se cruzan; tamaño ≠ significancia).
3. **Diagnosticar la violación del supuesto de hazards proporcionales** (punto 3.1): leer el test de Schoenfeld de Rossi (`age` p ≈ 0,0007), explicar por qué un **p bajo es malo** aquí y proponer la corrección (estratificar / interacción con el tiempo; **no** borrar).

## 13.9 — ¿Qué no se puede afirmar, y qué sigue en S14? Cierre (Sección 9 del cuaderno)

**Entregable de la sesión (`evaluacion/entregable.docx`).** Análisis de supervivencia de **churn** (curvas KM por segmento + Cox + CLV) con **recomendación** priorizada. Se materializa con `plantillas/reporte_supervivencia_clv.docx` y `plantillas/guia_schoenfeld.docx`; rúbrica vigesimal 0–20.

**Control corto de cierre.** Preguntas breves de interpretación (censura, `S(t)` como retención, lectura de HR, p bajo en Schoenfeld, CLV desde `S(t)`).

**Vínculo con el proyecto integrador.** El *tiempo-hasta-evento* y el **CLV** conectan con la logística de **S09** (churn como clasificación) y la causalidad de **S12**: pasar de *«¿quién se irá?»* a *«¿cuándo y cuánto vale intervenir?»*.

**Materiales de apoyo del cuaderno.** el cuaderno de la sesión (puente celda↔paso), la guía de supuestos de la sesión (fuente única de supuestos), la ficha de la sesión de réplica del paper (targets), el material de referencia de la sesión (recomputa y cruza contra el Excel).

### Para seguir explorando (fuentes de actualidad)
- **Eightx (29/05/2026)** — benchmark 2026 de churn de suscripción por periodo de facturación: `https://eightx.co/blog/average-ecommerce-subscription-churn-by-billing-period-2026`
- **Statsig (23/06/2025)** — supervivencia como métrica de producto (KM → hazard → Cox): `https://www.statsig.com/perspectives/survival-analysis-time-metrics`
- **IJLTEMAS 14(13), 2025** — KM + Cox para churn en telecom: `https://ideas.repec.org/a/bjb/journl/v14y2025i13p201-212.html`
- **arXiv 2510.11604 (2025/2026)** — supervivencia + IA explicable + segmentación para retención: `https://arxiv.org/abs/2510.11604`

### Bibliografía (verificada)
- **Cox, D. R. (1972).** *Regression Models and Life-Tables.* JRSS-B 34(2):187-220. DOI 10.1111/j.2517-6161.1972.tb00899.x
- **Kaplan, E. L. & Meier, P. (1958).** *Nonparametric Estimation from Incomplete Observations.* JASA 53(282):457-481. Copia docente: `https://web.stanford.edu/~lutian/coursepdf/KMpaper.pdf`
- **Schoenfeld, D. (1982).** *Partial Residuals for the Proportional Hazards Regression Model.* Biometrika 69(1):239-241.
- **Grambsch, P. M. & Therneau, T. M. (1994).** *Proportional Hazards Tests and Diagnostics Based on Weighted Residuals.* Biometrika 81(3):515-526.
- **Fader, Hardie & Lee (2005).** *RFM and CLV: Using Iso-Value Curves for Customer Base Analysis.* JMR 42(4):415-430. PDF: `http://www.brucehardie.com/papers/rfm_clv_2005-02-16.pdf`
- **James et al. (2023).** *ISLP*, cap. 11 «Survival Analysis and Censored Data». `https://www.statlearning.com/`
- **Therneau & Grambsch (2000).** *Modeling Survival Data: Extending the Cox Model.* Springer.

*Réplica reproducida con `lifelines` 0.30.3 (operativo = venv). Validación numérica en el material de referencia de la sesión (targets T1-T6 declarado para la sesión de réplica del paper).*